# AMPS I · Week 12 — Lab
## Solving saddle-point systems: Schur complement, block preconditioners, `fieldsplit`

| | |
|---|---|
| **Lecture sections** | Part A (Schur complement, ideal and practical block preconditioners, Darcy) · Part B (blocked/nested systems, `fieldsplit`) · Part C (measurements) · Part D (LLM) |
| **Time** | ≈ 80 min |
| **You will** | look at the spectrum of the Stokes matrix with and without block preconditioners · solve one problem three ways (monolithic, nested, blocked) · measure iterations vs $h$ for four solvers · time LU against MINRES in 2D and 3D · see why the "Stokes recipe" fails for Darcy |

Taylor–Hood, unit square, $\mu=1$, the Week 11 MMS $\tilde{\mathbf u}=(\sin\pi x\cos\pi y,\,-\cos\pi x\sin\pi y)$, $\tilde p=\sin\pi x\sin\pi y$. The pressure constant is
handled by a **null space** (not a pinned DOF) from §2 on.

**Reminder.** `%%fenicsx` cells run in a separate MPI process; ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.
Times measured on Colab vary between runs — compare trends, not the third digit.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The spectrum of the saddle-point matrix  *(lecture §A1–A4)*
Dense matrices on $8\times8$ (serial): interior velocity DOFs only, so $A$ is SPD. $S=BA^{-1}B^\top$; $\beta_h^2=\lambda_{\min}(S,M_p)$ on $L^2_0$.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import time
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def wall(t0):
    """Elapsed time, maximum over ranks (call on all ranks)."""
    return comm.allreduce(time.perf_counter() - t0, op=MPI.MAX)

import scipy.linalg as sla
import scipy.sparse as sp
import dolfinx.fem.petsc as fpetsc

def dense_blocks(N):
    """Dense A (interior velocity DOFs), B, Mp for Taylor-Hood on the N x N unit square, mu = 1 (serial)."""
    msh = mesh.create_unit_square(MPI.COMM_SELF, N, N)
    V = fem.functionspace(msh, ("Lagrange", 2, (2,))); Q = fem.functionspace(msh, ("Lagrange", 1))
    u, v, p, q = ufl.TrialFunction(V), ufl.TestFunction(V), ufl.TrialFunction(Q), ufl.TestFunction(Q)
    def dense(form):                       # PETSc assembly: fem.assemble_matrix (CSR) refuses the rectangular blocked B
        M = fpetsc.assemble_matrix(fem.form(form)); M.assemble()
        ai, aj, av = M.getValuesCSR(); s = M.getSize(); M.destroy()
        return sp.csr_matrix((av, aj, ai), shape=s).toarray()
    A = dense(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx)
    B = dense(-q * ufl.div(u) * ufl.dx)
    Mp = dense(p * q * ufl.dx)
    msh.topology.create_connectivity(1, 2)
    bd = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    I = np.setdiff1d(np.arange(A.shape[0]), np.concatenate([2 * bd, 2 * bd + 1]))   # interior velocity components
    return A[np.ix_(I, I)], B[:, I], Mp

A, B, Mp = dense_blocks(8)
n, m = A.shape[0], Mp.shape[0]
K = np.block([[A, B.T], [B, np.zeros((m, m))]])
ev = sla.eigvalsh(K)
tol = 1e-10 * np.abs(ev).max()
print(f"K: {n} velocity + {m} pressure unknowns; eigenvalues > 0: {np.sum(ev > tol)}, < 0: {np.sum(ev < -tol)}, = 0: {np.sum(np.abs(ev) <= tol)}")

S = B @ np.linalg.solve(A, B.T); S = 0.5 * (S + S.T)
sig = sla.eigvalsh(S, Mp); sig = sig[sig > 1e-10 * sig.max()]          # drop the constant pressure mode
print(f"eig(Mp^-1 S) in [{sig.min():.4f}, {sig.max():.4f}]  ->  beta_h = {np.sqrt(sig.min()):.4f}  (Week 11, Taylor-Hood 'right': 0.366)")

one = np.ones(m); Mo = Mp @ one
S_reg = S + np.outer(Mo, Mo) / (one @ Mo)                               # invertible; the constant stays a zero eigenvalue of K
out = {}
for name, P in (("none", np.eye(n + m)), ("diag_A_Mp", sla.block_diag(A, Mp)), ("diag_A_S", sla.block_diag(A, S_reg))):
    lam = np.real(sla.eigvals(K, P)); lam = lam[np.abs(lam) > 1e-8]
    out[name] = lam
    print(f"P = {name:9s}: negative in [{lam[lam < 0].min():8.4f}, {lam[lam < 0].max():8.4f}], positive in [{lam[lam > 0].min():.4f}, "
          f"{lam[lam > 0].max():.4f}], max|lam|/min|lam| = {np.abs(lam).max() / np.abs(lam).min():.4g}")
np.savez("/tmp/fnx/w12_spectra.npz", sig=sig, **out)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("/tmp/fnx/w12_spectra.npz")
fig, ax = plt.subplots(figsize=(9, 3))
for k, (name, lab, c) in enumerate((("none", "no preconditioner", "C3"), ("diag_A_Mp", "diag(A, Mp)", "C0"), ("diag_A_S", "diag(A, S)", "C2"))):
    ax.plot(d[name], np.full(d[name].shape, k), "|", color=c, ms=18)
ax.set_xscale("symlog", linthresh=1e-2); ax.set_yticks([0, 1, 2], ["no preconditioner", "diag(A, Mp)", "diag(A, S)"])
ax.set_ylim(-0.6, 2.6); ax.set_xlabel("eigenvalues of $P^{-1}K$ (symlog)"); fig.tight_layout(); plt.show()

> **✏️ Your turn (10 min).** Lecture §A4 claims: with $\mathscr P=\mathrm{diag}(A,M_p)$ every eigenvalue is $1$ or $\lambda=\frac{1\pm\sqrt{1+4\sigma}}{2}$ with $\sigma\in\operatorname{eig}(M_p^{-1}S)$.
> Check it: compute the predicted negative eigenvalues from `sig`, compare them (sorted) with the computed negative eigenvalues, and count how many computed eigenvalues equal 1.

In [ ]:
%%fenicsx -np 1
# TODO: compare (1 - sqrt(1 + 4 sigma))/2 with the negative eigenvalues of diag(A, Mp)^-1 K

## 2 · One problem, three assemblies  *(lecture §B1–B3)*
Monolithic (mixed element, MUMPS), **nested** (`kind="nest"`, `LinearProblem` with a preconditioner form `P`) and **blocked** (one matrix, index sets) — read
`stokes_blocked` and `solve_blocked` in the cell first.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import time
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def wall(t0):
    """Elapsed time, maximum over ranks (call on all ranks)."""
    return comm.allreduce(time.perf_counter() - t0, op=MPI.MAX)

U_EX = lambda x: ufl.as_vector([ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1]), -ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])])
P_EX = lambda x: ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])      # mean 4/pi^2

def stokes_blocked(N, dim=2, mu=1.0, pscale=None):
    """Taylor-Hood Stokes as a *blocked* system [[A, B^T], [B, 0]] with preconditioner matrix [[A, 0], [0, pscale*Mp]]
    (pscale defaults to 1/mu). dim=2: MMS on the unit square; dim=3: body force (0, 0, sin(pi x)) in the unit cube, u = 0 on the boundary.
    The pressure constant is handled by a null space. Returns a dict - keep it alive while you use A, P, b (it owns the mesh)."""
    msh = mesh.create_unit_square(comm, N, N) if dim == 2 else mesh.create_unit_cube(comm, N, N, N)
    V = fem.functionspace(msh, ("Lagrange", 2, (dim,)))
    Q = fem.functionspace(msh, ("Lagrange", 1))
    u, p = ufl.TrialFunction(V), ufl.TrialFunction(Q)
    v, q = ufl.TestFunction(V), ufl.TestFunction(Q)
    x = ufl.SpatialCoordinate(msh)
    muc = fem.Constant(msh, mu)
    ps = fem.Constant(msh, 1.0 / mu if pscale is None else pscale)
    a = fem.form([[muc * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, -p * ufl.div(v) * ufl.dx],
                  [-q * ufl.div(u) * ufl.dx, None]])
    a_p = fem.form([[muc * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, None],
                    [None, ps * p * q * ufl.dx]])
    msh.topology.create_connectivity(dim - 1, dim)
    bfacets = mesh.exterior_facet_indices(msh.topology)
    if dim == 2:
        f = -muc * ufl.div(ufl.grad(U_EX(x))) + ufl.grad(P_EX(x))
        ub = fem.Function(V); ub.interpolate(fem.Expression(U_EX(x), V.element.interpolation_points))
        bc = fem.dirichletbc(ub, fem.locate_dofs_topological(V, dim - 1, bfacets))
    else:
        f = ufl.as_vector([0, 0, ufl.sin(ufl.pi * x[0])])
        bc = fem.dirichletbc(np.zeros(dim), fem.locate_dofs_topological(V, dim - 1, bfacets), V)
    L = fem.form([ufl.inner(f, v) * ufl.dx, ufl.ZeroBaseForm((q,))])
    A = assemble_matrix(a, bcs=[bc]); A.assemble()
    P = assemble_matrix(a_p, bcs=[bc]); P.assemble()
    b = assemble_vector(L, kind=PETSc.Vec.Type.MPI)
    apply_lifting(b, a, bcs=fem.bcs_by_block(fem.extract_function_spaces(a, 1), [bc]))
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b, fem.bcs_by_block(fem.extract_function_spaces(L), [bc]))
    # index sets of the two fields in the blocked matrix
    nu = V.dofmap.index_map.size_local * dim
    off = V.dofmap.index_map.local_range[0] * dim + Q.dofmap.index_map.local_range[0]
    is_u = PETSc.IS().createStride(nu, off, 1, comm=comm)
    is_p = PETSc.IS().createStride(Q.dofmap.index_map.size_local, off + nu, 1, comm=comm)
    z = A.createVecLeft(); z.array[nu:] = 1.0; z.normalize()           # constant pressure, zero velocity
    nsp = PETSc.NullSpace().create(vectors=[z], comm=comm)
    A.setNullSpace(nsp); nsp.remove(b)
    return dict(msh=msh, V=V, Q=Q, A=A, P=P, b=b, is_u=is_u, is_p=is_p, nu=nu, dim=dim)

def solve_blocked(S, opts, prefix):
    """Solve the blocked system with PETSc options `opts` (fieldsplit names 'u' and 'p').
    Returns iterations, converged reason, wall time of setup + solve, solution vector."""
    o = PETSc.Options()
    for k, v in opts.items():
        o[prefix + k] = v
    ksp = PETSc.KSP().create(comm)
    ksp.setOptionsPrefix(prefix)
    lu = opts.get("pc_type") == "lu"
    ksp.setOperators(S["A"], S["A"] if lu else S["P"])
    if opts.get("pc_type") == "fieldsplit":
        ksp.getPC().setType("fieldsplit")
        ksp.getPC().setFieldSplitIS(("u", S["is_u"]), ("p", S["is_p"]))
    ksp.setFromOptions()
    if lu:
        ksp.getPC().setFactorSetUpSolverType()
        ksp.getPC().getFactorMatrix().setMumpsIcntl(icntl=24, ival=1)  # null-pivot detection: the pressure constant
        ksp.getPC().getFactorMatrix().setMumpsIcntl(icntl=25, ival=0)
    if opts.get("fieldsplit_u_pc_type") == "gamg":                     # GAMG reads the near-null space from the block size
        ksp.getPC().setUp()
        ksp.getPC().getFieldSplitSubKSP()[0].getPC().getOperators()[0].setBlockSize(S["dim"])
    x = S["A"].createVecRight()
    t0 = time.perf_counter()
    ksp.solve(S["b"], x)
    t = wall(t0)
    its, reason = ksp.getIterationNumber(), ksp.getConvergedReason()
    ksp.destroy()
    for k in opts:
        o.delValue(prefix + k)
    return its, reason, t, x

def velocity_error(S, x):
    """|u - u_h|_L2 for the 2D MMS."""
    u = fem.Function(S["V"]); u.x.array[:S["nu"]] = x.array_r[:S["nu"]]; u.x.scatter_forward()
    e = u - U_EX(ufl.SpatialCoordinate(S["msh"]))
    return np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(ufl.inner(e, e) * ufl.dx)), op=MPI.SUM))

FS = {"pc_type": "fieldsplit", "fieldsplit_u_ksp_type": "preonly", "fieldsplit_p_ksp_type": "preonly", "ksp_rtol": 1e-8, "ksp_max_it": 3000}
CFG = {
    "LU (MUMPS)":                      {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"},
    "MINRES, no PC":                   {"ksp_type": "minres", "pc_type": "none", "ksp_rtol": 1e-8, "ksp_max_it": 3000},
    "MINRES diag(LU, LU(Mp))":         {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "lu", "fieldsplit_p_pc_type": "lu"},
    "MINRES diag(BoomerAMG, Jac(Mp))": {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "hypre", "fieldsplit_p_pc_type": "jacobi"},
    "GMRES upper(BoomerAMG, Jac(Mp))": {**FS, "ksp_type": "gmres", "ksp_gmres_restart": 200, "pc_fieldsplit_type": "schur",
                                        "pc_fieldsplit_schur_fact_type": "upper", "pc_fieldsplit_schur_precondition": "a11",
                                        "fieldsplit_u_pc_type": "hypre", "fieldsplit_p_pc_type": "jacobi"},
    "MINRES diag(GAMG, Jac(Mp))":      {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "gamg", "fieldsplit_p_pc_type": "jacobi"},
}

import basix.ufl
N = 16
msh = mesh.create_unit_square(comm, N, N)
x = ufl.SpatialCoordinate(msh)
f = -ufl.div(ufl.grad(U_EX(x))) + ufl.grad(P_EX(x))
msh.topology.create_connectivity(1, 2)
bfac = mesh.exterior_facet_indices(msh.topology)
P2v = basix.ufl.element("Lagrange", "triangle", 2, shape=(2,)); P1 = basix.ufl.element("Lagrange", "triangle", 1)
L2 = lambda e: np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(ufl.inner(e, e) * ufl.dx)), op=MPI.SUM))

# (a) monolithic: Week 11, but the constant is left to MUMPS (null-pivot detection) instead of a pin
W = fem.functionspace(msh, basix.ufl.mixed_element([P2v, P1])); W0, _ = W.sub(0).collapse()
(uu, pp), (vv, qq) = ufl.TrialFunctions(W), ufl.TestFunctions(W)
ubW = fem.Function(W0); ubW.interpolate(fem.Expression(U_EX(x), W0.element.interpolation_points))
bcW = fem.dirichletbc(ubW, fem.locate_dofs_topological((W.sub(0), W0), 1, bfac), W.sub(0))
wh = LinearProblem((ufl.inner(ufl.grad(uu), ufl.grad(vv)) - pp * ufl.div(vv) - qq * ufl.div(uu)) * ufl.dx, ufl.inner(f, vv) * ufl.dx,
                   bcs=[bcW], petsc_options_prefix="lab_mono_",
                   petsc_options={"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps",
                                  "mat_mumps_icntl_24": 1, "mat_mumps_icntl_25": 0}).solve()
u_mono = wh.sub(0).collapse()
say(f"monolithic (MUMPS)       : |u - u_h| = {L2(u_mono - U_EX(x)):.4e}")

# (b) nested: two spaces, a list of forms, kind="nest", preconditioner P = diag(A, Mp)
V = fem.functionspace(msh, P2v); Q = fem.functionspace(msh, P1)
u, p, v, q = ufl.TrialFunction(V), ufl.TrialFunction(Q), ufl.TestFunction(V), ufl.TestFunction(Q)
ub = fem.Function(V); ub.interpolate(fem.Expression(U_EX(x), V.element.interpolation_points))
bcV = fem.dirichletbc(ub, fem.locate_dofs_topological(V, 1, bfac))
a_bl = [[ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, -p * ufl.div(v) * ufl.dx], [-q * ufl.div(u) * ufl.dx, None]]
L_bl = [ufl.inner(f, v) * ufl.dx, ufl.ZeroBaseForm((q,))]
P_bl = [[a_bl[0][0], None], [None, p * q * ufl.dx]]
probN = LinearProblem(a_bl, L_bl, bcs=[bcV], P=P_bl, kind="nest", petsc_options_prefix="lab_nest_",
                      petsc_options={"ksp_type": "minres", "ksp_rtol": 1e-10, "pc_type": "fieldsplit", "pc_fieldsplit_type": "additive",
                                     "fieldsplit_0_ksp_type": "preonly", "fieldsplit_0_pc_type": "hypre",
                                     "fieldsplit_1_ksp_type": "preonly", "fieldsplit_1_pc_type": "jacobi"})
z = fem.petsc.create_vector([V, Q], kind="nest"); zs = z.getNestSubVecs(); zs[0].set(0.0); zs[1].set(1.0); z.normalize()
probN.A.setNullSpace(PETSc.NullSpace().create(vectors=[z]))
u_nest, p_nest = probN.solve()
say(f"nested  (MINRES)         : its {probN.solver.getIterationNumber()}, reason {probN.solver.getConvergedReason()}, "
    f"|u - u_h| = {L2(u_nest - U_EX(x)):.4e}, |u_nest - u_mono| = {L2(u_nest - u_mono):.1e}")

# (c) blocked: the helpers above
S = stokes_blocked(16)
its, r, t, xb = solve_blocked(S, CFG["MINRES diag(BoomerAMG, Jac(Mp))"], "lab_blk_")
say(f"blocked (MINRES + IS)    : its {its}, reason {r}, |u - u_h| = {velocity_error(S, xb):.4e}")
its, r, t, xb = solve_blocked(S, CFG["LU (MUMPS)"], "lab_blklu_")
say(f"blocked (LU)             : reason {r}, |u - u_h| = {velocity_error(S, xb):.4e}")

> **✏️ Your turn (5 min).** Solve the **nested** problem with GMRES and the block **upper-triangular** Schur preconditioner instead
> (`pc_fieldsplit_type schur`, `pc_fieldsplit_schur_fact_type upper`, `pc_fieldsplit_schur_precondition a11` — the $(1,1)$ block of `P` is $M_p$).
> Compare the iteration count with MINRES. Why may MINRES **not** be used with this preconditioner?

In [ ]:
%%fenicsx -np 2
# TODO: nested system, GMRES + fieldsplit schur upper

## 3 · Iterations vs $h$  *(lecture §A5, §C2)*
Four solvers on the blocked system, $N=8,\dots,64$, rtol $10^{-8}$. Prediction first: which counts stay flat?

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import time
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def wall(t0):
    """Elapsed time, maximum over ranks (call on all ranks)."""
    return comm.allreduce(time.perf_counter() - t0, op=MPI.MAX)

U_EX = lambda x: ufl.as_vector([ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1]), -ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])])
P_EX = lambda x: ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])      # mean 4/pi^2

def stokes_blocked(N, dim=2, mu=1.0, pscale=None):
    """Taylor-Hood Stokes as a *blocked* system [[A, B^T], [B, 0]] with preconditioner matrix [[A, 0], [0, pscale*Mp]]
    (pscale defaults to 1/mu). dim=2: MMS on the unit square; dim=3: body force (0, 0, sin(pi x)) in the unit cube, u = 0 on the boundary.
    The pressure constant is handled by a null space. Returns a dict - keep it alive while you use A, P, b (it owns the mesh)."""
    msh = mesh.create_unit_square(comm, N, N) if dim == 2 else mesh.create_unit_cube(comm, N, N, N)
    V = fem.functionspace(msh, ("Lagrange", 2, (dim,)))
    Q = fem.functionspace(msh, ("Lagrange", 1))
    u, p = ufl.TrialFunction(V), ufl.TrialFunction(Q)
    v, q = ufl.TestFunction(V), ufl.TestFunction(Q)
    x = ufl.SpatialCoordinate(msh)
    muc = fem.Constant(msh, mu)
    ps = fem.Constant(msh, 1.0 / mu if pscale is None else pscale)
    a = fem.form([[muc * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, -p * ufl.div(v) * ufl.dx],
                  [-q * ufl.div(u) * ufl.dx, None]])
    a_p = fem.form([[muc * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, None],
                    [None, ps * p * q * ufl.dx]])
    msh.topology.create_connectivity(dim - 1, dim)
    bfacets = mesh.exterior_facet_indices(msh.topology)
    if dim == 2:
        f = -muc * ufl.div(ufl.grad(U_EX(x))) + ufl.grad(P_EX(x))
        ub = fem.Function(V); ub.interpolate(fem.Expression(U_EX(x), V.element.interpolation_points))
        bc = fem.dirichletbc(ub, fem.locate_dofs_topological(V, dim - 1, bfacets))
    else:
        f = ufl.as_vector([0, 0, ufl.sin(ufl.pi * x[0])])
        bc = fem.dirichletbc(np.zeros(dim), fem.locate_dofs_topological(V, dim - 1, bfacets), V)
    L = fem.form([ufl.inner(f, v) * ufl.dx, ufl.ZeroBaseForm((q,))])
    A = assemble_matrix(a, bcs=[bc]); A.assemble()
    P = assemble_matrix(a_p, bcs=[bc]); P.assemble()
    b = assemble_vector(L, kind=PETSc.Vec.Type.MPI)
    apply_lifting(b, a, bcs=fem.bcs_by_block(fem.extract_function_spaces(a, 1), [bc]))
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b, fem.bcs_by_block(fem.extract_function_spaces(L), [bc]))
    # index sets of the two fields in the blocked matrix
    nu = V.dofmap.index_map.size_local * dim
    off = V.dofmap.index_map.local_range[0] * dim + Q.dofmap.index_map.local_range[0]
    is_u = PETSc.IS().createStride(nu, off, 1, comm=comm)
    is_p = PETSc.IS().createStride(Q.dofmap.index_map.size_local, off + nu, 1, comm=comm)
    z = A.createVecLeft(); z.array[nu:] = 1.0; z.normalize()           # constant pressure, zero velocity
    nsp = PETSc.NullSpace().create(vectors=[z], comm=comm)
    A.setNullSpace(nsp); nsp.remove(b)
    return dict(msh=msh, V=V, Q=Q, A=A, P=P, b=b, is_u=is_u, is_p=is_p, nu=nu, dim=dim)

def solve_blocked(S, opts, prefix):
    """Solve the blocked system with PETSc options `opts` (fieldsplit names 'u' and 'p').
    Returns iterations, converged reason, wall time of setup + solve, solution vector."""
    o = PETSc.Options()
    for k, v in opts.items():
        o[prefix + k] = v
    ksp = PETSc.KSP().create(comm)
    ksp.setOptionsPrefix(prefix)
    lu = opts.get("pc_type") == "lu"
    ksp.setOperators(S["A"], S["A"] if lu else S["P"])
    if opts.get("pc_type") == "fieldsplit":
        ksp.getPC().setType("fieldsplit")
        ksp.getPC().setFieldSplitIS(("u", S["is_u"]), ("p", S["is_p"]))
    ksp.setFromOptions()
    if lu:
        ksp.getPC().setFactorSetUpSolverType()
        ksp.getPC().getFactorMatrix().setMumpsIcntl(icntl=24, ival=1)  # null-pivot detection: the pressure constant
        ksp.getPC().getFactorMatrix().setMumpsIcntl(icntl=25, ival=0)
    if opts.get("fieldsplit_u_pc_type") == "gamg":                     # GAMG reads the near-null space from the block size
        ksp.getPC().setUp()
        ksp.getPC().getFieldSplitSubKSP()[0].getPC().getOperators()[0].setBlockSize(S["dim"])
    x = S["A"].createVecRight()
    t0 = time.perf_counter()
    ksp.solve(S["b"], x)
    t = wall(t0)
    its, reason = ksp.getIterationNumber(), ksp.getConvergedReason()
    ksp.destroy()
    for k in opts:
        o.delValue(prefix + k)
    return its, reason, t, x

def velocity_error(S, x):
    """|u - u_h|_L2 for the 2D MMS."""
    u = fem.Function(S["V"]); u.x.array[:S["nu"]] = x.array_r[:S["nu"]]; u.x.scatter_forward()
    e = u - U_EX(ufl.SpatialCoordinate(S["msh"]))
    return np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(ufl.inner(e, e) * ufl.dx)), op=MPI.SUM))

FS = {"pc_type": "fieldsplit", "fieldsplit_u_ksp_type": "preonly", "fieldsplit_p_ksp_type": "preonly", "ksp_rtol": 1e-8, "ksp_max_it": 3000}
CFG = {
    "LU (MUMPS)":                      {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"},
    "MINRES, no PC":                   {"ksp_type": "minres", "pc_type": "none", "ksp_rtol": 1e-8, "ksp_max_it": 3000},
    "MINRES diag(LU, LU(Mp))":         {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "lu", "fieldsplit_p_pc_type": "lu"},
    "MINRES diag(BoomerAMG, Jac(Mp))": {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "hypre", "fieldsplit_p_pc_type": "jacobi"},
    "GMRES upper(BoomerAMG, Jac(Mp))": {**FS, "ksp_type": "gmres", "ksp_gmres_restart": 200, "pc_fieldsplit_type": "schur",
                                        "pc_fieldsplit_schur_fact_type": "upper", "pc_fieldsplit_schur_precondition": "a11",
                                        "fieldsplit_u_pc_type": "hypre", "fieldsplit_p_pc_type": "jacobi"},
    "MINRES diag(GAMG, Jac(Mp))":      {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "gamg", "fieldsplit_p_pc_type": "jacobi"},
}

names = ["MINRES, no PC", "MINRES diag(LU, LU(Mp))", "MINRES diag(BoomerAMG, Jac(Mp))", "GMRES upper(BoomerAMG, Jac(Mp))"]
say(f"{'N':>4s} {'DOFs':>7s} | " + " | ".join(names))
for N in (8, 16, 32, 64):
    S = stokes_blocked(N)
    row = []
    for name in names:
        its, r, t, xs = solve_blocked(S, CFG[name], f"lab3_{N}_")
        row.append(f"{its:5d}" + ("" if r > 0 else f" (reason {r})"))
    say(f"{N:4d} {S['A'].getSize()[0]:7d} | " + " | ".join(f"{c:>{len(nm)}s}" for c, nm in zip(row, names)))

> **✏️ Your turn (5 min).** Viscosity $\mu=10^{-2}$ and $10^{-4}$, $N=32$, MINRES diag(BoomerAMG, Jacobi): once with the scaled block $M_p/\mu$ (`pscale=None`) and once
> with the unscaled $M_p$ (`pscale=1.0`). Report iterations and $\|\mathbf u-\mathbf u_h\|$. Explain with §A4.

In [ ]:
%%fenicsx -np 2
# TODO: viscosity scaling of the pressure block

## 4 · Time vs DOFs in 2D and 3D  *(lecture §C4, §C6)*
KSP setup + solve only (assembly and JIT compilation excluded). 2D MMS $N=32,64,128$; 3D unit cube $N=4,8,12$ (body force, $\mathbf u=0$ on the boundary).
About 20 s on the build machine; longer on Colab.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import time
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def wall(t0):
    """Elapsed time, maximum over ranks (call on all ranks)."""
    return comm.allreduce(time.perf_counter() - t0, op=MPI.MAX)

U_EX = lambda x: ufl.as_vector([ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1]), -ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])])
P_EX = lambda x: ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])      # mean 4/pi^2

def stokes_blocked(N, dim=2, mu=1.0, pscale=None):
    """Taylor-Hood Stokes as a *blocked* system [[A, B^T], [B, 0]] with preconditioner matrix [[A, 0], [0, pscale*Mp]]
    (pscale defaults to 1/mu). dim=2: MMS on the unit square; dim=3: body force (0, 0, sin(pi x)) in the unit cube, u = 0 on the boundary.
    The pressure constant is handled by a null space. Returns a dict - keep it alive while you use A, P, b (it owns the mesh)."""
    msh = mesh.create_unit_square(comm, N, N) if dim == 2 else mesh.create_unit_cube(comm, N, N, N)
    V = fem.functionspace(msh, ("Lagrange", 2, (dim,)))
    Q = fem.functionspace(msh, ("Lagrange", 1))
    u, p = ufl.TrialFunction(V), ufl.TrialFunction(Q)
    v, q = ufl.TestFunction(V), ufl.TestFunction(Q)
    x = ufl.SpatialCoordinate(msh)
    muc = fem.Constant(msh, mu)
    ps = fem.Constant(msh, 1.0 / mu if pscale is None else pscale)
    a = fem.form([[muc * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, -p * ufl.div(v) * ufl.dx],
                  [-q * ufl.div(u) * ufl.dx, None]])
    a_p = fem.form([[muc * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, None],
                    [None, ps * p * q * ufl.dx]])
    msh.topology.create_connectivity(dim - 1, dim)
    bfacets = mesh.exterior_facet_indices(msh.topology)
    if dim == 2:
        f = -muc * ufl.div(ufl.grad(U_EX(x))) + ufl.grad(P_EX(x))
        ub = fem.Function(V); ub.interpolate(fem.Expression(U_EX(x), V.element.interpolation_points))
        bc = fem.dirichletbc(ub, fem.locate_dofs_topological(V, dim - 1, bfacets))
    else:
        f = ufl.as_vector([0, 0, ufl.sin(ufl.pi * x[0])])
        bc = fem.dirichletbc(np.zeros(dim), fem.locate_dofs_topological(V, dim - 1, bfacets), V)
    L = fem.form([ufl.inner(f, v) * ufl.dx, ufl.ZeroBaseForm((q,))])
    A = assemble_matrix(a, bcs=[bc]); A.assemble()
    P = assemble_matrix(a_p, bcs=[bc]); P.assemble()
    b = assemble_vector(L, kind=PETSc.Vec.Type.MPI)
    apply_lifting(b, a, bcs=fem.bcs_by_block(fem.extract_function_spaces(a, 1), [bc]))
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b, fem.bcs_by_block(fem.extract_function_spaces(L), [bc]))
    # index sets of the two fields in the blocked matrix
    nu = V.dofmap.index_map.size_local * dim
    off = V.dofmap.index_map.local_range[0] * dim + Q.dofmap.index_map.local_range[0]
    is_u = PETSc.IS().createStride(nu, off, 1, comm=comm)
    is_p = PETSc.IS().createStride(Q.dofmap.index_map.size_local, off + nu, 1, comm=comm)
    z = A.createVecLeft(); z.array[nu:] = 1.0; z.normalize()           # constant pressure, zero velocity
    nsp = PETSc.NullSpace().create(vectors=[z], comm=comm)
    A.setNullSpace(nsp); nsp.remove(b)
    return dict(msh=msh, V=V, Q=Q, A=A, P=P, b=b, is_u=is_u, is_p=is_p, nu=nu, dim=dim)

def solve_blocked(S, opts, prefix):
    """Solve the blocked system with PETSc options `opts` (fieldsplit names 'u' and 'p').
    Returns iterations, converged reason, wall time of setup + solve, solution vector."""
    o = PETSc.Options()
    for k, v in opts.items():
        o[prefix + k] = v
    ksp = PETSc.KSP().create(comm)
    ksp.setOptionsPrefix(prefix)
    lu = opts.get("pc_type") == "lu"
    ksp.setOperators(S["A"], S["A"] if lu else S["P"])
    if opts.get("pc_type") == "fieldsplit":
        ksp.getPC().setType("fieldsplit")
        ksp.getPC().setFieldSplitIS(("u", S["is_u"]), ("p", S["is_p"]))
    ksp.setFromOptions()
    if lu:
        ksp.getPC().setFactorSetUpSolverType()
        ksp.getPC().getFactorMatrix().setMumpsIcntl(icntl=24, ival=1)  # null-pivot detection: the pressure constant
        ksp.getPC().getFactorMatrix().setMumpsIcntl(icntl=25, ival=0)
    if opts.get("fieldsplit_u_pc_type") == "gamg":                     # GAMG reads the near-null space from the block size
        ksp.getPC().setUp()
        ksp.getPC().getFieldSplitSubKSP()[0].getPC().getOperators()[0].setBlockSize(S["dim"])
    x = S["A"].createVecRight()
    t0 = time.perf_counter()
    ksp.solve(S["b"], x)
    t = wall(t0)
    its, reason = ksp.getIterationNumber(), ksp.getConvergedReason()
    ksp.destroy()
    for k in opts:
        o.delValue(prefix + k)
    return its, reason, t, x

def velocity_error(S, x):
    """|u - u_h|_L2 for the 2D MMS."""
    u = fem.Function(S["V"]); u.x.array[:S["nu"]] = x.array_r[:S["nu"]]; u.x.scatter_forward()
    e = u - U_EX(ufl.SpatialCoordinate(S["msh"]))
    return np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(ufl.inner(e, e) * ufl.dx)), op=MPI.SUM))

FS = {"pc_type": "fieldsplit", "fieldsplit_u_ksp_type": "preonly", "fieldsplit_p_ksp_type": "preonly", "ksp_rtol": 1e-8, "ksp_max_it": 3000}
CFG = {
    "LU (MUMPS)":                      {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"},
    "MINRES, no PC":                   {"ksp_type": "minres", "pc_type": "none", "ksp_rtol": 1e-8, "ksp_max_it": 3000},
    "MINRES diag(LU, LU(Mp))":         {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "lu", "fieldsplit_p_pc_type": "lu"},
    "MINRES diag(BoomerAMG, Jac(Mp))": {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "hypre", "fieldsplit_p_pc_type": "jacobi"},
    "GMRES upper(BoomerAMG, Jac(Mp))": {**FS, "ksp_type": "gmres", "ksp_gmres_restart": 200, "pc_fieldsplit_type": "schur",
                                        "pc_fieldsplit_schur_fact_type": "upper", "pc_fieldsplit_schur_precondition": "a11",
                                        "fieldsplit_u_pc_type": "hypre", "fieldsplit_p_pc_type": "jacobi"},
    "MINRES diag(GAMG, Jac(Mp))":      {**FS, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                        "fieldsplit_u_pc_type": "gamg", "fieldsplit_p_pc_type": "jacobi"},
}

rec = []
for dim, Ns, it_name in ((2, (32, 64, 128), "MINRES diag(BoomerAMG, Jac(Mp))"), (3, (4, 8, 12), "MINRES diag(GAMG, Jac(Mp))")):
    for N in Ns:
        S = stokes_blocked(N, dim=dim)
        n = S["A"].getSize()[0]
        _, r1, t1, x1 = solve_blocked(S, CFG["LU (MUMPS)"], f"lab4lu{dim}{N}_")
        its, r2, t2, x2 = solve_blocked(S, CFG[it_name], f"lab4it{dim}{N}_")
        d = x2.copy(); d.axpy(-1.0, x1)
        say(f"{dim}D N={N:3d} DOFs={n:7d}: LU {t1:6.2f} s (reason {r1}) | {it_name}: {t2:6.2f} s, {its} its (reason {r2}) | "
            f"|x_it - x_LU|/|x_LU| = {d.norm() / x1.norm():.1e}")
        rec.append((dim, n, t1, t2))
if comm.rank == 0:
    np.save("/tmp/fnx/w12_times.npy", np.array(rec))

> **✏️ Your turn (5 min).** Plot time vs DOFs (log–log) for both solvers in 2D and 3D and fit the slopes over the last two points. Where would the 3D curves cross
> if you could not see it? What does the agreement $|x_{it}-x_{LU}|/|x_{LU}|$ tell you, and why must you check it before comparing times?

In [ ]:
# TODO: log-log plot of the timings in /tmp/fnx/w12_times.npy and slopes

## 5 · Darcy: the Stokes recipe fails  *(lecture §A6, §C5)*
Mixed Poisson, RT1 × DG0, $K=1$. Nested system. Two preconditioners are given: L² block diagonal $\mathrm{diag}(M_\sigma,M_u)$ and the algebraic Schur complement `selfp`.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import time
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def wall(t0):
    """Elapsed time, maximum over ranks (call on all ranks)."""
    return comm.allreduce(time.perf_counter() - t0, op=MPI.MAX)

def darcy(N, P, opts, prefix):
    """Mixed Poisson (Darcy) with RT1 x DG0 on the unit square, K = 1, u = sin(pi x) sin(pi y) (u = 0 on the boundary is natural).
    P: None (preconditioner = operator), "L2" or "Hdiv" (block-diagonal preconditioner matrix). Nested system, LinearProblem.
    Returns iterations, reason, |u - u_h|, |sigma - sigma_h|, DOFs."""
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("RT", 1)); W = fem.functionspace(msh, ("DG", 0))
    s, u = ufl.TrialFunction(V), ufl.TrialFunction(W)
    t, v = ufl.TestFunction(V), ufl.TestFunction(W)
    x = ufl.SpatialCoordinate(msh)
    uex = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    sex = -ufl.grad(uex)
    a = [[ufl.inner(s, t) * ufl.dx, -u * ufl.div(t) * ufl.dx],
         [-ufl.div(s) * v * ufl.dx, None]]
    L = [ufl.ZeroBaseForm((t,)), -ufl.div(sex) * v * ufl.dx]
    Pf = {None: None,
          "L2":   [[ufl.inner(s, t) * ufl.dx, None], [None, u * v * ufl.dx]],
          "Hdiv": [[ufl.inner(s, t) * ufl.dx + ufl.div(s) * ufl.div(t) * ufl.dx, None], [None, u * v * ufl.dx]]}[P]
    pr = LinearProblem(a, L, P=Pf, kind="nest" if opts.get("pc_type") == "fieldsplit" else None,
                       petsc_options_prefix=prefix, petsc_options=opts)
    sh, uh = pr.solve()
    I = lambda e: np.sqrt(comm.allreduce(fem.assemble_scalar(fem.form(e * ufl.dx)), op=MPI.SUM))
    return (pr.solver.getIterationNumber(), pr.solver.getConvergedReason(), I((uh - uex) ** 2), I(ufl.inner(sh - sex, sh - sex)),
            V.dofmap.index_map.size_global + W.dofmap.index_map.size_global)

FD = {"pc_type": "fieldsplit", "fieldsplit_0_ksp_type": "preonly", "fieldsplit_1_ksp_type": "preonly", "ksp_rtol": 1e-8, "ksp_max_it": 5000}

cfg = {"L2 diag (MINRES)": ("L2", {**FD, "ksp_type": "minres", "pc_fieldsplit_type": "additive",
                                   "fieldsplit_0_pc_type": "lu", "fieldsplit_1_pc_type": "jacobi"}),
       "Schur upper selfp + AMG (GMRES)": (None, {**FD, "ksp_type": "gmres", "pc_fieldsplit_type": "schur",
                                                  "pc_fieldsplit_schur_fact_type": "upper", "pc_fieldsplit_schur_precondition": "selfp",
                                                  "fieldsplit_0_pc_type": "jacobi", "fieldsplit_1_pc_type": "hypre"})}
for N in (16, 32, 64):
    row = []
    for name, (P, o) in cfg.items():
        its, r, eu, es, n = darcy(N, P, o, f"lab5_{N}_{len(row)}_")
        row.append(f"{name}: {its}" + ("" if r > 0 else f" (reason {r})"))
    say(f"N={N:3d} DOFs={n:6d} | " + " | ".join(row) + f" | |u-u_h| {eu:.2e}, |sigma-sigma_h| {es:.2e}")

> **✏️ Your turn (10 min).** Add the **H(div)** block-diagonal preconditioner ($P$ = `"Hdiv"`: $M_\sigma+(\nabla\cdot\boldsymbol\sigma,\nabla\cdot\boldsymbol\tau)$ and $M_u$, MINRES,
> LU on block 0, Jacobi on block 1). Iterations for $N=16,32,64$? Explain in two sentences why $M_\sigma$ alone is the wrong block here but $A$ alone was right for Stokes.

In [ ]:
%%fenicsx -np 2
# TODO: H(div) block-diagonal preconditioner for Darcy

## 6 · LLM-assisted coding: live demo  *(lecture §D1)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, one %%fenicsx -np 2 cell. Taylor-Hood Stokes on the unit square, manufactured solution
u = (sin(pi x) cos(pi y), -cos(pi x) sin(pi y)), p = sin(pi x) sin(pi y), mu = 1, Dirichlet velocity on the whole boundary.
Assemble a blocked system (two function spaces, a list of forms) and solve it with MINRES preconditioned by a block-diagonal
PETSc fieldsplit: one BoomerAMG V-cycle for the velocity block and Jacobi on the pressure mass matrix. Handle the pressure constant
with a null space. For N = 16, 32, 64 print the iteration count, the KSP converged reason and the L2 velocity error.
```
Paste the result below **unchanged**, run it, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] The preconditioner matrix has $M_p$ (scaled by $1/\mu$) in the $(1,1)$ block — the operator's $(1,1)$ block is empty
- [ ] The pressure constant is handled (null space)
- [ ] Iterations roughly constant in $N$
- [ ] The velocity errors are Week 11's: $9.7\times10^{-5}$, $1.2\times10^{-5}$, $1.5\times10^{-6}$ for $N=16,32,64$ (rate 3)